In [2]:
import pandas as pd
import numpy as np

city_path = "../data/city_safety/city_safety_2023_geo.csv"
cities_df = pd.read_csv(city_path)

print("City columns:", cities_df.columns.tolist())
print(cities_df.head())

print(cities_df.shape)

City columns: ['Rank', 'city', 'country', 'crime_index', 'safety_index', 'year', 'city_norm', 'country_norm', 'lat', 'lon']
   Rank          city            country  crime_index  safety_index  year  \
0     1       Caracas          Venezuela         83.6          16.4  2023   
1     2      Pretoria       South Africa         82.0          18.0  2023   
2     3        Durban       South Africa         81.0          19.0  2023   
3     4  Port Moresby   Papua New Guinea         80.7          19.3  2023   
4     5  Johannesburg       South Africa         80.7          19.3  2023   

      city_norm      country_norm        lat         lon  
0       caracas         venezuela  10.506093  -66.914601  
1      pretoria      south africa -25.745928   28.187910  
2        durban      south africa -29.861825   31.009909  
3  port moresby  papua new guinea  -9.474330  147.159950  
4  johannesburg      south africa -26.205000   28.049722  
(416, 10)


In [3]:
# Keep rows with valid coordinates and metrics
cities_df = cities_df.dropna(subset=["lat", "lon", "crime_index", "safety_index"]).copy()

# Clean country strings
cities_df["country"] = cities_df["country"].str.strip()
cities_df["country_norm"] = cities_df["country"].str.lower()

# Map 2-letter state/province codes to country
state_to_country = {
    "ak": "united states",
    "ca": "united states",
    "co": "united states",
    "fl": "united states",
    "ga": "united states",
    "hi": "united states",
    "id": "united states",
    "il": "united states",
    "in": "united states",
    "ky": "united states",
    "la": "united states",
    "ma": "united states",
    "mi": "united states",
    "mn": "united states",
    "mo": "united states",
    "nc": "united states",
    "nm": "united states",
    "nv": "united states",
    "ny": "united states",
    "oh": "united states",
    "or": "united states",
    "pa": "united states",
    "tx": "united states",
    "ut": "united states",
    "wa": "united states",
    "wi": "united states",
    "dc": "united states",
    "az": "united states",
    "md": "united states",
    "tn": "united states",
    "ab": "canada",
    "bc": "canada"
}

cities_df["country_norm"] = cities_df["country_norm"].replace(state_to_country)

# Handle specific name fixes 
name_fixes = {
    "colombia": "colombia",
    "dominican republic": "dominican republic",
    "canada": "canada",
    "thailand": "thailand",
    "moldova": "moldova",
    "georgia": "georgia",
    "hong kong (china)": "hong kong",
}

cities_df["country_norm"] = cities_df["country_norm"].replace(name_fixes)

print(cities_df.shape)

(416, 10)


In [4]:
cities_df.head()

,Rank,city,country,crime_index,safety_index,year,city_norm,country_norm,lat,lon
0,1,Caracas,Venezuela,83.6,16.4,2023,caracas,venezuela,10.506093,-66.914601
1,2,Pretoria,South Africa,82.0,18.0,2023,pretoria,south africa,-25.745928,28.187910
2,3,Durban,South Africa,81.0,19.0,2023,durban,south africa,-29.861825,31.009909
3,4,Port Moresby,Papua New Guinea,80.7,19.3,2023,port moresby,papua new guinea,-9.474330,147.159950
4,5,Johannesburg,South Africa,80.7,19.3,2023,johannesburg,south africa,-26.205000,28.049722


In [5]:
with pd.option_context("display.max_rows", None,
                       "display.max_columns", None,
                       "display.width", 200):
    print(cities_df)

     Rank                               city                 country  crime_index  safety_index  year                          city_norm            country_norm        lat         lon
0       1                            Caracas               Venezuela         83.6          16.4  2023                            caracas               venezuela  10.506093  -66.914601
1       2                           Pretoria            South Africa         82.0          18.0  2023                           pretoria            south africa -25.745928   28.187910
2       3                             Durban            South Africa         81.0          19.0  2023                             durban            south africa -29.861825   31.009909
3       4                       Port Moresby        Papua New Guinea         80.7          19.3  2023                       port moresby        papua new guinea  -9.474330  147.159950
4       5                       Johannesburg            South Africa         80.

In [7]:
out_path = "../data/city_safety/city_safety_2023_geo_clean.csv"
cities_df.to_csv(out_path, index=False)
print("Saved cleaned city data to:", out_path, "rows:", len(cities_df))

Saved cleaned city data to: ../data/city_safety/city_safety_2023_geo_clean.csv rows: 416


In [8]:
wci_2022_path = "../data/city_safety/world_crime_index_2022_clean.csv"
wci_2022 = pd.read_csv(wci_2022_path)

print("2022 columns:", wci_2022.columns.tolist())
print(wci_2022.head())
print("Rows 2022:", len(wci_2022))

# Quick overlap check
geo_keys = set(
    cities_df[["city", "country_norm"]]
    .assign(city=lambda df: df["city"].str.lower().str.strip())
    .itertuples(index=False, name=None)
)
wci_keys = set(
    wci_2022[["city", "country_norm"]]
    .assign(city=lambda df: df["city"].str.lower().str.strip())
    .itertuples(index=False, name=None)
)

print("Common city-country pairs:", len(geo_keys & wci_keys))
print("Only in 2022:", len(wci_keys - geo_keys))
print("Only in 2023_geo:", len(geo_keys - wci_keys))

2022 columns: ['rank', 'city', 'crime_index', 'safety_index', 'country', 'city_norm', 'country_norm']
   rank            city  crime_index  safety_index           country  \
0     1         Caracas        83.98         16.02         Venezuela   
1     2        Pretoria        81.98         18.02      South Africa   
2     3          Celaya        81.80         18.20            Mexico   
3     4  San Pedro Sula        80.87         19.13          Honduras   
4     5    Port Moresby        80.71         19.29  Papua New Guinea   

        city_norm      country_norm  
0         caracas         venezuela  
1        pretoria      south africa  
2          celaya            mexico  
3  san pedro sula          honduras  
4    port moresby  papua new guinea  
Rows 2022: 453
Common city-country pairs: 360
Only in 2022: 93
Only in 2023_geo: 56


In [10]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd

wci_2022 = pd.read_csv("../data/city_safety/world_crime_index_2022_clean.csv")

# Build unique (city, country) pairs to geocode
places_22 = (
    wci_2022[["city", "country"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

geolocator = Nominatim(user_agent="travel_safety_capstone_2022")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def geocode_place(row):
    query = f"{row['city']}, {row['country']}"
    loc = geocode(query)
    if loc is None:
        return pd.Series({"lat": None, "lon": None})
    return pd.Series({"lat": loc.latitude, "lon": loc.longitude})

geo_results_22 = places_22.apply(geocode_place, axis=1)
places_22_geo = pd.concat([places_22, geo_results_22], axis=1)

# Drop failed geocodes
places_22_geo = places_22_geo.dropna(subset=["lat", "lon"]).reset_index(drop=True)

print(places_22_geo.head(), places_22_geo.shape)

# Merge lat/lon back into wci_2022
wci_2022_geo = wci_2022.merge(
    places_22_geo,
    on=["city", "country"],
    how="inner"
)

print(wci_2022_geo.head(), wci_2022_geo.shape)

             city           country        lat         lon
0         Caracas         Venezuela  10.506093  -66.914601
1        Pretoria      South Africa -25.745928   28.187910
2          Celaya            Mexico  20.522285 -100.830774
3  San Pedro Sula          Honduras  15.505354  -88.025084
4    Port Moresby  Papua New Guinea  -9.474330  147.159950 (453, 4)
   rank            city  crime_index  safety_index           country  \
0     1         Caracas        83.98         16.02         Venezuela   
1     2        Pretoria        81.98         18.02      South Africa   
2     3          Celaya        81.80         18.20            Mexico   
3     4  San Pedro Sula        80.87         19.13          Honduras   
4     5    Port Moresby        80.71         19.29  Papua New Guinea   

        city_norm      country_norm        lat         lon  
0         caracas         venezuela  10.506093  -66.914601  
1        pretoria      south africa -25.745928   28.187910  
2          celaya    

OSError: Cannot save file into a non-existent directory: 'data\city_safety'

In [11]:
wci_2022_geo.to_csv("../data/city_safety/world_crime_index_2022_geo_clean.csv", index=False)

2023 rows: 416
2022_geo rows: 453
